# RetailIQ Exploratory Data Analysis

**Project context:** Brazilian E-Commerce Public Dataset by Olist.

This notebook is designed as a professional EDA deliverable for an e-commerce business analyst workflow. It focuses on business questions, data quality, demand patterns, customer behavior, regional performance, and product performance.

The notebook prefers the cleaned feature tables in `data/processed/` and falls back to the raw Olist CSVs in `data/raw/` when processed outputs are not available.

## 1. Dataset Overview

**Business question:** What tables are available, what is their grain, and how complete are they?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titleweight'] = 'bold'

ROOT = Path.cwd().resolve()
while ROOT.name != 'RetailIQ' and ROOT.parent != ROOT:
    ROOT = ROOT.parent

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = (
        result.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'[^0-9a-zA-Z]+', '_', regex=True)
        .str.replace(r'_+', '_', regex=True)
        .str.strip('_')
    )
    return result

def read_table(*paths: Path) -> pd.DataFrame:
    for path in paths:
        if path.exists():
            return standardize_columns(pd.read_csv(path, low_memory=False))
    raise FileNotFoundError(f'None of the candidate files exist: {paths}')

def load_table(processed_name: str, raw_name: str) -> pd.DataFrame:
    processed_path = PROCESSED_DIR / processed_name
    raw_path = RAW_DIR / raw_name
    return read_table(processed_path, raw_path)

tables = {}
tables['orders'] = load_table('fact_orders_features.csv', 'olist_orders_dataset.csv')
tables['order_items'] = load_table('fact_order_items_features.csv', 'olist_order_items_dataset.csv')
tables['customers'] = load_table('dim_customers_features.csv', 'olist_customers_dataset.csv')
tables['products'] = load_table('dim_products_features.csv', 'olist_products_dataset.csv')
tables['sellers'] = load_table('dim_sellers_features.csv', 'olist_sellers_dataset.csv')
tables['payments'] = load_table('order_payments_clean.csv', 'olist_order_payments_dataset.csv')
tables['reviews'] = load_table('order_reviews_clean.csv', 'olist_order_reviews_dataset.csv')
tables['categories'] = load_table('category_translation_clean.csv', 'product_category_name_translation.csv')

overview = []
for name, df in tables.items():
    overview.append({
        'table': name,
        'rows': len(df),
        'columns': len(df.columns),
        'missing_cells': int(df.isna().sum().sum()),
        'missing_pct': round(df.isna().sum().sum() / max(df.size, 1) * 100, 2),
    })

overview_df = pd.DataFrame(overview).sort_values('rows', ascending=False).reset_index(drop=True)
display(overview_df)

## 2. Missing Values

**Business question:** Which tables and fields would most likely distort KPI reporting if left uncleaned?

In [ ]:
missing_summary = []
for name, df in tables.items():
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    top_missing = missing_pct[missing_pct > 0].head(8)
    for column, pct in top_missing.items():
        missing_summary.append({'table': name, 'column': column, 'missing_pct': round(float(pct), 2)})

missing_df = pd.DataFrame(missing_summary).sort_values(['missing_pct', 'table'], ascending=[False, True])
display(missing_df.head(20))

if not missing_df.empty:
    top_missing_plot = missing_df.head(15)
    plt.figure(figsize=(12, 7))
    sns.barplot(data=top_missing_plot, y='table', x='missing_pct', hue='column', dodge=False, palette='Blues_r')
    plt.title('Which fields have the highest missing-rate risk for KPI reporting?')
    plt.xlabel('Missing values (%)')
    plt.ylabel('Table')
    plt.legend(title='Column', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 3. Distributions

**Business question:** How are order value, freight cost, review score, and product size distributed?

In [ ]:
order_df = tables['orders'].copy()
item_df = tables['order_items'].copy()
product_df = tables['products'].copy()
review_df = tables['reviews'].copy()

if 'order_value' not in order_df.columns:
    order_totals = item_df.groupby('order_id', as_index=False).agg(order_value=('price', 'sum'), freight_value=('freight_value', 'sum'))
    order_df = order_df.merge(order_totals, on='order_id', how='left')

product_numeric_cols = [c for c in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'] if c in product_df.columns]
for col in ['order_value', 'freight_value', 'review_score'] + product_numeric_cols:
    if col in order_df.columns:
        order_df[col] = pd.to_numeric(order_df[col], errors='coerce')
    if col in item_df.columns:
        item_df[col] = pd.to_numeric(item_df[col], errors='coerce')
    if col in review_df.columns:
        review_df[col] = pd.to_numeric(review_df[col], errors='coerce')
    if col in product_df.columns:
        product_df[col] = pd.to_numeric(product_df[col], errors='coerce')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.histplot(order_df['order_value'].dropna(), bins=40, kde=True, ax=axes[0, 0], color='#1f77b4')
axes[0, 0].set_title('What is the distribution of order value?')
axes[0, 0].set_xlabel('Order value')

sns.histplot(item_df['freight_value'].dropna(), bins=40, kde=True, ax=axes[0, 1], color='#ff7f0e')
axes[0, 1].set_title('How concentrated is shipping cost across items?')
axes[0, 1].set_xlabel('Freight value')

if 'review_score' in review_df.columns:
    sns.countplot(x='review_score', data=review_df, ax=axes[1, 0], palette='Greens')
    axes[1, 0].set_title('How do customers rate completed orders?')
    axes[1, 0].set_xlabel('Review score')
else:
    axes[1, 0].text(0.5, 0.5, 'review_score not available', ha='center', va='center')
    axes[1, 0].set_axis_off()

if product_numeric_cols:
    sns.histplot(product_df[product_numeric_cols[0]].dropna(), bins=40, kde=True, ax=axes[1, 1], color='#9467bd')
    axes[1, 1].set_title('What is the distribution of product size / weight?')
    axes[1, 1].set_xlabel(product_numeric_cols[0])
else:
    axes[1, 1].text(0.5, 0.5, 'No numeric product size column available', ha='center', va='center')
    axes[1, 1].set_axis_off()

plt.tight_layout()
plt.show()

## 4. Correlation

**Business question:** Which numeric drivers move together and may explain order economics or service quality?

In [ ]:
corr_candidates = []
for df in [tables['orders'], tables['order_items'], tables['products'], tables['reviews']]:
    numeric_df = df.select_dtypes(include=['number']).copy()
    if not numeric_df.empty:
        corr_candidates.append(numeric_df)

corr_df = pd.concat(corr_candidates, axis=1) if corr_candidates else pd.DataFrame()
corr_df = corr_df.loc[:, ~corr_df.columns.duplicated()]
corr_matrix = corr_df.corr(numeric_only=True)

if not corr_matrix.empty:
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False, linewidths=0.2)
    plt.title('Which numeric metrics move together?')
    plt.tight_layout()
    plt.show()
else:
    display(Markdown('No numeric columns available for correlation analysis.'))

## 5. Time Trends

**Business question:** Is demand growing, flattening, or seasonal over time?

In [ ]:
orders_time = tables['orders'].copy()
if 'order_purchase_timestamp' in orders_time.columns:
    orders_time['order_purchase_timestamp'] = pd.to_datetime(orders_time['order_purchase_timestamp'], errors='coerce')
    if 'order_value' not in orders_time.columns and 'order_id' in item_df.columns:
        order_totals = item_df.groupby('order_id', as_index=False).agg(order_value=('price', 'sum'))
        orders_time = orders_time.merge(order_totals, on='order_id', how='left')
    orders_time['order_month'] = orders_time['order_purchase_timestamp'].dt.to_period('M').astype(str)
    monthly_trend = orders_time.groupby('order_month', as_index=False).agg(orders=('order_id', 'nunique'), revenue=('order_value', 'sum'))

    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax1.plot(monthly_trend['order_month'], monthly_trend['revenue'], marker='o', color='#1f77b4', label='Revenue')
    ax1.set_ylabel('Revenue')
    ax1.set_xlabel('Month')
    ax1.tick_params(axis='x', rotation=45)
    ax1.set_title('How does revenue evolve over time?')
    ax2 = ax1.twinx()
    ax2.plot(monthly_trend['order_month'], monthly_trend['orders'], marker='s', color='#ff7f0e', label='Orders')
    ax2.set_ylabel('Orders')
    fig.tight_layout()
    plt.show()

    if 'order_purchase_timestamp' in orders_time.columns:
        weekday_trend = orders_time.assign(weekday=orders_time['order_purchase_timestamp'].dt.day_name())
        weekday_counts = weekday_trend['weekday'].value_counts().reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
        plt.figure(figsize=(12, 5))
        sns.barplot(x=weekday_counts.index, y=weekday_counts.values, color='#2ca02c')
        plt.title('Which days of the week generate the most orders?')
        plt.ylabel('Orders')
        plt.xlabel('Weekday')
        plt.tight_layout()
        plt.show()
else:
    display(Markdown('order_purchase_timestamp not available for time-trend analysis.'))

## 6. Regional Analysis

**Business question:** Which states drive the highest revenue and order volume?

In [ ]:
regional_orders = tables['orders'].copy()
regional_customers = tables['customers'].copy()
if 'order_purchase_timestamp' in regional_orders.columns:
    regional_orders['order_purchase_timestamp'] = pd.to_datetime(regional_orders['order_purchase_timestamp'], errors='coerce')

if 'order_value' not in regional_orders.columns:
    order_totals = item_df.groupby('order_id', as_index=False).agg(order_value=('price', 'sum'))
    regional_orders = regional_orders.merge(order_totals, on='order_id', how='left')

regional = regional_orders.merge(regional_customers[['customer_id', 'customer_state']], on='customer_id', how='left')
state_summary = regional.groupby('customer_state', as_index=False).agg(revenue=('order_value', 'sum'), orders=('order_id', 'nunique'))
state_summary = state_summary.sort_values('revenue', ascending=False).head(15)

plt.figure(figsize=(12, 7))
sns.barplot(data=state_summary, y='customer_state', x='revenue', color='#1f77b4')
plt.title('Which states generate the most revenue?')
plt.xlabel('Revenue')
plt.ylabel('State')
plt.tight_layout()
plt.show()

display(state_summary)

## 7. Customer Analysis

**Business question:** Are repeat customers driving a meaningful share of revenue?

In [ ]:
customer_base = tables['customers'].copy()
customer_orders = tables['orders'].copy()
if 'order_purchase_timestamp' in customer_orders.columns:
    customer_orders['order_purchase_timestamp'] = pd.to_datetime(customer_orders['order_purchase_timestamp'], errors='coerce')

if 'order_value' not in customer_orders.columns:
    order_totals = item_df.groupby('order_id', as_index=False).agg(order_value=('price', 'sum'))
    customer_orders = customer_orders.merge(order_totals, on='order_id', how='left')

customer_metrics = customer_orders.groupby('customer_id', as_index=False).agg(total_orders=('order_id', 'nunique'), revenue=('order_value', 'sum'))
customer_metrics = customer_metrics.merge(customer_base[['customer_id', 'customer_unique_id']], on='customer_id', how='left')
customer_metrics['repeat_customer'] = customer_metrics['total_orders'] > 1
repeat_share = customer_metrics['repeat_customer'].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.histplot(customer_metrics['revenue'].dropna(), bins=40, kde=True, ax=axes[0], color='#9467bd')
axes[0].set_title('How concentrated is customer revenue?')
axes[0].set_xlabel('Revenue per customer')

repeat_counts = customer_metrics['repeat_customer'].value_counts().rename(index={False: 'One-time', True: 'Repeat'})
axes[1].pie(repeat_counts.values, labels=repeat_counts.index, autopct='%1.1f%%', startangle=90, colors=['#ff7f0e', '#2ca02c'])
axes[1].set_title('What share of customers are repeat buyers?')

plt.suptitle(f'Repeat customer share: {repeat_share:.1%}', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

display(customer_metrics.sort_values('revenue', ascending=False).head(10))

## 8. Product Analysis

**Business question:** Which product categories create the most value and which deserve operational attention?

In [ ]:
product_items = tables['order_items'].copy()
product_master = tables['products'].copy()
category_lookup = tables['categories'].copy()

product_summary = product_items.merge(product_master[['product_id', 'product_category_name']], on='product_id', how='left')
product_summary = product_summary.merge(category_lookup, on='product_category_name', how='left')
category_col = 'product_category_name_english' if 'product_category_name_english' in product_summary.columns else 'product_category_name'
product_summary['item_revenue'] = product_summary['price'].fillna(0) + product_summary['freight_value'].fillna(0)
category_summary = product_summary.groupby(category_col, as_index=False).agg(revenue=('item_revenue', 'sum'), orders=('order_id', 'nunique'), units_sold=('product_id', 'count')).sort_values('revenue', ascending=False).head(15)

plt.figure(figsize=(12, 7))
sns.barplot(data=category_summary, y=category_col, x='revenue', palette='viridis')
plt.title('Which product categories contribute the most revenue?')
plt.xlabel('Revenue')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

display(category_summary)

## 9. Business Insights

**Business question:** What are the most actionable findings an e-commerce team should act on first?

The cell below generates a compact insight summary from the analyzed tables. Update the narrative after the notebook is executed against the latest data.

In [ ]:
insights = []
total_revenue = customer_metrics['revenue'].sum() if 'customer_metrics' in globals() else np.nan
top_state = state_summary.iloc[0]['customer_state'] if not state_summary.empty else 'N/A'
top_category = category_summary.iloc[0][category_col] if not category_summary.empty else 'N/A'
repeat_rate = repeat_share if 'repeat_share' in globals() else np.nan

insights.append(f'Total revenue observed in the dataset: {total_revenue:,.2f}')
insights.append(f'Top revenue state: {top_state}')
insights.append(f'Top revenue category: {top_category}')
insights.append(f'Repeat customer share: {repeat_rate:.1%}' if pd.notna(repeat_rate) else 'Repeat customer share: not available')

display(Markdown('### Draft Insights'))
for item in insights:
    display(Markdown(f'- {item}'))

display(Markdown('### Recommended actions'))
display(Markdown('- Prioritize high-revenue states and categories in assortment and marketing.'))
display(Markdown('- Reduce operational friction in states with slower or less efficient delivery patterns.'))
display(Markdown('- Build retention campaigns for one-time customers with high first-order value.'))